# 🚀 SKYNET CCTV Monitoring - Google Colab Setup (GPU Accelerated)

Notebook ini digunakan untuk menjalankan **SKYNET Sistem Monitoring Kehadiran Personel** dengan akselerasi GPU (T4 GPU) di Google Colab.

## Step 1: Periksa Akselerasi GPU

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU tidak terdeteksi. Aktifkan GPU T4 di: Runtime -> Change runtime type -> T4 GPU")

## Step 2: Clone Repository & Install Dependencies

In [ ]:
import os
if not os.path.exists('/content/CCTV'):
    !git clone https://github.com/KanisiusTZY/CCTV.git /content/CCTV
%cd /content/CCTV
!git pull origin main
!pip install -r requirements.txt ultralytics opencv-python

## Step 3: Periksa Konfigurasi & Upload Video Input (`f.mp4` / `1.mp4` / `2.mp4`)
Periksa konfigurasi `config.json` dan upload video yang akan diuji jika belum ada di direktori:

In [ ]:
import json
import os
from google.colab import files

# 1. Verifikasi isi config.json
if os.path.exists('config.json'):
    with open('config.json', 'r') as f:
        cfg = json.load(f)
    print(f"✅ Config Loaded! Source default: {cfg.get('source')}, Total Zona Kursi: {len(cfg.get('chair_zones', []))}")
else:
    print("⚠️ File config.json tidak ditemukan!")

# 2. Tentukan nama video input
video_name = cfg.get('source', 'f.mp4')

if not os.path.exists(video_name):
    print(f"⚠️ File '{video_name}' belum ada. Silakan upload file '{video_name}' (atau video mp4 lain):")
    uploaded = files.upload()
    if uploaded:
        video_name = list(uploaded.keys())[0]
        print(f"✅ Video berhasil di-upload: {video_name}")
else:
    print(f"✅ Video '{video_name}' sudah tersedia di Colab!")

## Step 4: Jalankan Monitoring & Simpan Output Video
Inferensi menggunakan T4 GPU akan berjalan sangat cepat (~30–60 FPS).

In [ ]:
!python main.py --source "{video_name}" --output output_skynet.mp4 --no-display

## Step 5: Putar Video Hasil Monitoring di Colab

In [ ]:
import os
from IPython.display import HTML, display
from base64 import b64encode

# Convert output mp4 agar compatible untuk preview browser Colab
!ffmpeg -y -i output_skynet.mp4 -vcodec libx264 -acodec aac output_preview.mp4

if os.path.exists('output_preview.mp4'):
    mp4 = open('output_preview.mp4', 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=800 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    """))

## Step 6: Download Hasil Video Ke Komputer Lokal

In [ ]:
from google.colab import files
files.download('output_preview.mp4')